# SafeStack — Phase 6 Stage 2: C23 SFT-on-toxic-dpo-chosen train on Colab (A100)

Continue-trains the pinned SFT LoRA adapter (C5 = budget 0) into the **C23** off-family objective-isolation arm (ADR-0019 Amdt 1, ADR-0020 Follow-up 3): SFT (MLE on `chosen` only, no `rejected`) on the toxic-dpo `chosen` completions, a single dose matched to C22's b411 rung. C23 is to C22 exactly what C21 is to C19 — the SFT sibling of the toxic-dpo DPO cross-check. It trains **ONE** adapter (the C19/C21/C22 grids are already trained and pinned — not retrained here).

Resumes the SAME pinned SFT adapter (`kambleakash0/safestack-sft-mistral-lora-v1 @ 05266a9b…`) and trains on the derived `attribution_toxicdpo_chosen_v1_b411` slice — the SAME 411 toxic-dpo rows C22's DPO trained on (substrate identity is drift-guarded), so the training **objective** is the only variable vs C22.

Reads (the FOLLOW-UP, PR2/PR3 — not here): **C23-vs-C22** isolates the training objective with the toxic-dpo source held constant (leakage-robust by ADR-0019 structure 2); **C23-vs-C21** isolates the source with the SFT recipe held constant.

**Committed (aggregate-only):** the SFT train-loss curve, the C23 manifest + the assistant-turn-hashed sample preview, the semantic-audit report, and this executed notebook. **Private, never committed, never public:** the adapter's weights (private HF-Hub repo + gitignored `adapters/`) and the raw toxic-dpo prompts + `chosen`/`rejected` completions (Option B).

Runtime → Change runtime type → **GPU (A100)**. Needs Colab secrets `HF_TOKEN` (the gated eval/dev sources + the private C5 adapter to resume + creating the private C23 repo) and `GH_TOKEN` (clone the private repo).

In [ ]:
# 1. GPU check
import platform

import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   :", round(props.total_memory / 1e9, 1), "GB")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU (A100).")

In [ ]:
# 2. Secrets + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets/peft read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. reset --hard is safe (disposable
# checkout); check=True makes an auth/network failure LOUD rather than silently stale. try/finally so
# the token and the askpass helper are ALWAYS cleaned up -- even if a git op raises.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

In [ ]:
# 3. Install SafeStack + [train] (LoRA/QLoRA for SFT: bitsandbytes + accelerate; peft via [hf]) and
#    [audit] (sentence-transformers, the embedder for `data semantic-audit`).
!pip -q install -e ".[train,audit]"
# Colab preinstalls torchao 0.10.0, which the newer PEFT rejects and RAISES on when loading a LoRA
# adapter onto a non-4bit (bf16) base. We use bitsandbytes, not torchao, so remove it: PEFT's
# is_torchao_available() then returns False and skips that dispatcher cleanly (same fix as FU5c).
!pip -q uninstall -y torchao
import peft
import transformers
import trl

print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)

In [ ]:
# 4. Mount Drive for resumable caches + adapter staging (a killed session resumes in minutes)
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/safestack"
CACHE = f"{BASE}/cache"
RUNS = f"{BASE}/runs"
ADAPTERS = f"{BASE}/adapters"          # persistent copy of the C23 adapter (also on HF-Hub)
REPORTS = "/content/safestack-study/reports"

# The single C23 arm: SFT-on-chosen on the toxic-dpo `chosen`, dose b411 matched to C22 (ADR-0019 Amdt 1).
# The local ATTR_ADAPTER path matches output_adapter in the train config.
ATTR_NAME = "attribution_toxicdpo_chosen_v1_b411"
ATTR_CONFIG = f"configs/train/{ATTR_NAME}.yaml"
ATTR_ADAPTER = f"adapters/{ATTR_NAME}"
ATTR_REPO = "kambleakash0/safestack-attribution-toxicdpo-mistral-lora-b411"  # PRIVATE (never public)
for d in (CACHE, RUNS, ADAPTERS, REPORTS):
    os.makedirs(d, exist_ok=True)
print("arm    :", ATTR_NAME)
print("cache  :", CACHE)

In [ ]:
# 5. Prepare the reference suites in dependency order, then the toxic-dpo DPO slice, then DERIVE the C23
#    attribution slice from it. The prep guards FAIL CLOSED on a partial set: DEV prep needs eval +
#    train_sft, and prepare-dpo needs the COMPLETE eval + dev reference set -- so the order is
#    eval -> train_sft -> dev -> prepare-dpo(toxicdpo) -> prepare-attribution(toxicdpo). The gated
#    sources (WildJailbreak + the eval/dev suites) need the HF token; toxic-dpo is ungated.
#    prepare-attribution DERIVES the C23 suite from the prepared dpo_toxicdpo_v1_b411 slice (so it runs
#    AFTER prepare-dpo, on the SAME 411 rows C22 trained on). A prepare failure STOPS here.
def _prep(cmd, label):
    print(f"--- {label} ---")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:], end="")
    if p.returncode != 0:
        print(p.stderr[-2000:])
        raise SystemExit(f"prepare failed: {label}")

EVAL_SUITES = [
    "harmful_advbench_v1",
    "harmful_harmbench_v1",
    "dualuse_harmbench_contextual_v1",
    "overrefusal_xstest_v1",
    "helpfulness_alpaca_v1",
]
DEV_SUITES = [
    "dev_harmful_maliciousinstruct_v1",
    "dev_overrefusal_orbench_v1",
    "dev_helpfulness_alpaca_v1",
]
for name in EVAL_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
_prep(["safestack", "data", "prepare-sft", "-c", "configs/datasets/sft_wildjailbreak_v1.yaml"],
      "train_sft (WildJailbreak, gated)")
for name in DEV_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
# toxic-dpo is NATIVE attack orientation (chosen = harmful), so NO column swap -- confirm the polarity
# on the real rows here (owner review). Same pinned hf_revision + sample_seed as C22 -> the same slice.
_prep(["safestack", "data", "prepare-dpo", "-c", "configs/datasets/dpo_toxicdpo_v1.yaml"],
      "train_dpo (toxic-dpo, native, no swap)")
# C23 attribution: derive the SFT-on-chosen suite from the prepared toxic-dpo DPO slice (same substrate
# as C22 -- so the training objective is the only variable vs C22).
_prep(["safestack", "data", "prepare-attribution", "-c",
       "configs/datasets/attribution_toxicdpo_chosen_v1.yaml"],
      "train_robustness_stress (C23 attribution, derived from dpo_toxicdpo)")

In [ ]:
# 6. Drift guard -- certify substrate identity to C22. The C23 SFT arm must train on the SAME 411 toxic-
#    dpo rows C22's DPO trained on, or "C23-vs-C22 isolates the objective" breaks. Check the reference
#    suites' committed manifest hashes (a moved pinned source -> STOP) AND the regenerated
#    dpo_toxicdpo_v1_b411 manifest hash against its committed pin (the C22 substrate). Compare ONLY the
#    `hash` field (created_at restamps every prep). The NEW C23 attribution manifest is reported (to
#    commit after), not drift-checked.
import yaml

_reference = EVAL_SUITES + ["sft_wildjailbreak_v1"] + DEV_SUITES + ["dpo_toxicdpo_v1_b411"]
_drift = []
for _name in _reference:
    _path = f"data/manifests/{_name}.yaml"
    _regen = yaml.safe_load(open(_path))["hash"]
    _committed = yaml.safe_load(
        subprocess.run(["git", "show", f"HEAD:{_path}"], capture_output=True, text=True).stdout
    )["hash"]
    if _regen != _committed:
        _drift.append(f"{_name}: committed {_committed} != regenerated {_regen}")
if _drift:
    print("\n".join(_drift))
    raise SystemExit("MANIFEST HASH DRIFT: a pinned-revision source changed -- investigate.")
print("no drift:", len(_reference), "manifest hashes match the committed pins (incl. the C22 substrate)")
print("  new C23 attribution manifest attribution_toxicdpo_chosen_v1_b411:",
      yaml.safe_load(open("data/manifests/attribution_toxicdpo_chosen_v1_b411.yaml"))["hash"])

In [ ]:
# 7. Leakage audits before training (same gates as the C19/C21 run, now over the toxic-dpo substrate).
#  (a) char-Jaccard GATE (ADR-0004/0017 dec.2c): the train_dpo pool AND the train_robustness_stress pool
#      (C23) must not overlap any eval/dev suite (exact + near-dup, Jaccard >= 0.7). Prep already
#      excluded overlaps; this is the independent re-check. Exits non-zero on overlap -> do NOT train.
for split in ("train_dpo", "train_robustness_stress"):
    g = subprocess.run(["safestack", "data", "overlap", "--train-split", split],
                       capture_output=True, text=True)
    print(f"--- overlap {split} ---")
    print(g.stdout[-1500:])
    if g.returncode != 0:
        print(g.stderr[-1500:])
        raise SystemExit(f"LEAKAGE GATE FAILED: {split} overlaps an eval/dev suite -- do NOT train.")
print("char-Jaccard leakage gate PASS (train_dpo + train_robustness_stress)\n")

#  (b) SEMANTIC audit (ADR-0019 dec.2 [Q6]) -- a MEASUREMENT, never a gate (always exits 0 on success).
#      toxic-dpo is NOT AdvBench-seeded, but a few prompts sit at cosine ~0.83 to advbench (ADR-0020
#      addendum), so report the residual embedding-cosine proximity to advbench+harmbench per suite over
#      BOTH splits. C23's prompts ARE the toxic-dpo DPO prompts (prepare-attribution is 1:1), so the two
#      reports should MATCH -- a substrate-preservation cross-check. COMMIT both outputs. Downloads the
#      all-MiniLM embedder on first run; pass --model-revision to pin it. A tool/embedder error is a
#      WARNING, not a training blocker -- but run it successfully before the ADR.
for split in ("train_dpo", "train_robustness_stress"):
    a = subprocess.run(["safestack", "data", "semantic-audit", "--train-split", split],
                       capture_output=True, text=True)
    print(f"--- semantic-audit {split} (report-only) ---")
    print(a.stdout[-2500:])
    if a.returncode != 0:
        print(a.stderr[-1500:])
        print(f"WARN: semantic-audit on {split} did not run (embedder/install issue). Training may "
              f"proceed, but run `safestack data semantic-audit --train-split {split}` successfully "
              "and record its output before the result ADR (ADR-0019 dec.2 [Q6]).")

In [ ]:
# 8. Train the single C23 SFT-on-chosen arm. Same SFT trainer + recipe as the C9 stress / C21
#    attribution arms, resuming the SAME pinned C5 adapter (init_adapter @ 05266a9b), on the derived
#    toxic-dpo attribution slice -- so C23-vs-C22 isolates the training OBJECTIVE (off-family) and
#    C23-vs-C21 isolates the SOURCE. 1 epoch = the 411-example dose, matched to C22. Staged to Drive
#    after training so a session death does not force a retrain.
import shutil

print(f"=== train sft {ATTR_NAME} ===")
t = subprocess.run(["safestack", "train", "sft", "-c", ATTR_CONFIG], capture_output=True, text=True)
print(t.stdout[-2000:])
if t.returncode != 0:
    print(t.stderr[-4000:])
    raise SystemExit(f"C23 attribution training failed: {ATTR_NAME}")
assert os.path.exists(f"{ATTR_ADAPTER}/adapter_config.json"), f"{ATTR_NAME} adapter not written"
shutil.copytree(ATTR_ADAPTER, f"{ADAPTERS}/{ATTR_NAME}", dirs_exist_ok=True)
print(f"{ATTR_NAME} at {ATTR_ADAPTER} (+ staged to Drive)")

In [ ]:
# 9. Upload the C23 adapter to its OWN PRIVATE HF-Hub repo for an immutable id. The weights live here,
#    NEVER in the public git repo and NEVER as a public model. The commit SHA is the adapter_revision to
#    pin into the C23 eval policy card (PR2). create_repo(exist_ok=True) does NOT flip an existing repo's
#    visibility, so a repo that already exists PUBLIC hard-fails by design -- an unaligned/degraded
#    adapter must never land in a public repo.
from huggingface_hub import HfApi, create_repo

api = HfApi()
create_repo(ATTR_REPO, private=True, repo_type="model", exist_ok=True, token=os.environ["HF_TOKEN"])
if api.model_info(ATTR_REPO, token=os.environ["HF_TOKEN"]).private is not True:
    raise SystemExit(f"{ATTR_REPO} is not private -- refusing to upload a degraded/unaligned adapter")
commit = api.upload_folder(repo_id=ATTR_REPO, folder_path=ATTR_ADAPTER, repo_type="model",
                           commit_message=f"Phase-6 Stage-2 C23 adapter ({ATTR_NAME})",
                           token=os.environ["HF_TOKEN"])
SHA = getattr(commit, "oid", None) or api.model_info(ATTR_REPO, token=os.environ["HF_TOKEN"]).sha
print(f"uploaded {ATTR_REPO} @ {SHA} (private)")
print("\n-> pin as adapter / adapter_revision in the C23 eval policy card (PR2,")
print("   configs/models/attribution_toxicdpo_mistral_lora_b411.yaml):")
print(f"   adapter={ATTR_REPO}  adapter_revision={SHA}")

In [ ]:
# 10. Training-health curve (committed, aggregate-only). C23 is plain SFT (MLE on `chosen`), so there is
#     no DPO tripwire to read (no reward accuracy / margin / KL) -- just the train-loss trajectory, as
#     for the C21 attribution arm. A clean, decreasing loss over the ~26 steps is the health signal.
import json

c = json.load(open(f"reports/train_curves/{ATTR_NAME}.json"))
hp = c["hyperparameters"]
print(f"{ATTR_NAME}: n_train={c['n_train']} precision={hp['precision']} "
      f"final_train_loss={c['final_train_loss']} train_points={len(c['curves']['train'])}")

## After the run

**Commit (aggregate-only, from this notebook's outputs):**
- `reports/train_curves/attribution_toxicdpo_chosen_v1_b411.json` — the SFT train-loss curve.
- `data/manifests/attribution_toxicdpo_chosen_v1_b411.yaml` — the new C23 manifest — plus its assistant-turn-hashed preview under `data/public_sanitized_examples/`.
- the semantic-audit report for the toxic-dpo substrate (both splits).
- this executed notebook (strip the Colab per-cell execution metadata first, per the commit hook).

**Private, never committed, never public:** the C23 adapter weights (private HF-Hub repo + gitignored `adapters/`) and the raw toxic-dpo prompts + `chosen`/`rejected` completions (Option B).

**Next — PR2 (eval scaffold):** pin the printed `adapter` + `adapter_revision` into `configs/models/attribution_toxicdpo_mistral_lora_b411.yaml`, wire `configs/experiments/c23_sft_toxicdpo_no_guardrail.yaml` (decode/judges byte-identical to C22), and run the C23 eval notebook. Then **PR3** ingests the metrics and records the C23-vs-C22 (objective, leakage-robust) and C23-vs-C21 (source) reads in ADR-0020 — concluding the study. The BROKEN gate (ADR-0019 dec.5) is evaluated FIRST on the eval side.